# 🛸 ESP32-S3 TinyML Drone Acoustic Classifier Trainer (100% Free Colab Alternative)

This notebook is a **completely free, cloud-GPU accelerated alternative** to Edge Impulse.
It trains an **ultra-lightweight INT8 Quantized Convolutional Neural Network (Conv1D)** on 1-second audio recordings of drones vs. environmental noise, and exports a C header file (`model_data.h`) for the ESP32-S3.

### Pipeline Steps:
1. **Load & Slice Audio**: Ingests 16 kHz WAV files into 1.0s (16,000-sample) windows.
2. **Extract MFE Features**: Computes 40 Mel-Frequency Energy bands across 99 time frames.
3. **Train Conv1D Model**: Trains a compact neural network in Keras.
4. **INT8 Quantization**: Quantizes weights and activations to 8-bit integers for ESP-NN hardware acceleration.
5. **Export to C Header**: Generates `model_data.h` ready to flash to the ESP32-S3.

In [ ]:
# Step 1: Install Dependencies
!pip install -q tensorflow librosa numpy matplotlib

In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Available:", tf.config.list_physical_devices('GPU'))

### Step 2: Upload Audio Dataset or Generate Synthetic Audio
You can upload your `dataset.zip` (containing `drone/` and `background/` folders) or run the synthetic generator below to test the pipeline.

In [ ]:
# Create synthetic drone (harmonic whine) and background noise (pink/white noise) for demonstration
SAMPLE_RATE = 16000
WINDOW_SAMPLES = 16000 # 1.0 second

def generate_demo_dataset(num_samples=200):
    X = []
    y = []
    t = np.linspace(0, 1.0, WINDOW_SAMPLES, endpoint=False)
    
    for _ in range(num_samples // 2):
        # Class 1: Drone (fundamental ~400-800Hz + motor harmonics + minor noise)
        f0 = np.random.uniform(350, 850)
        drone = 0.5 * np.sin(2 * np.pi * f0 * t) + \
                0.3 * np.sin(2 * np.pi * 2 * f0 * t) + \
                0.2 * np.sin(2 * np.pi * 3 * f0 * t) + \
                0.1 * np.random.normal(0, 0.1, WINDOW_SAMPLES)
        X.append(drone.astype(np.float32))
        y.append(1)
        
        # Class 0: Background (wind rumble + environmental noise)
        f_wind = np.random.uniform(30, 120)
        bg = 0.6 * np.sin(2 * np.pi * f_wind * t) + \
             0.3 * np.random.normal(0, 0.3, WINDOW_SAMPLES)
        X.append(bg.astype(np.float32))
        y.append(0)
        
    return np.array(X), np.array(y)

X_raw, y_raw = generate_demo_dataset(400)
print(f"Generated {len(X_raw)} raw 1.0s audio samples.")

### Step 3: Mel-Frequency Energy (MFE) Feature Extraction
Computes 40 Mel filterbanks with frame length 20ms (320 samples) and frame stride 10ms (160 samples).

In [ ]:
def extract_mfe_features(audio_batch):
    # STFT: frame_length=320, frame_step=160, fft_length=512
    stfts = tf.signal.stft(audio_batch, frame_length=320, frame_step=160, fft_length=512)
    spectrograms = tf.abs(stfts)
    
    # Mel filterbank: 40 bands between 150 Hz and 4000 Hz
    num_spectrogram_bins = stfts.shape[-1]
    linear_to_mel_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=40,
        num_spectrogram_bins=num_spectrogram_bins,
        sample_rate=SAMPLE_RATE,
        lower_edge_hertz=150.0,
        upper_edge_hertz=4000.0
    )
    mel_spectrograms = tf.tensordot(spectrograms, linear_to_mel_matrix, 1)
    mel_spectrograms.set_shape(spectrograms.shape[:-1].concatenate(linear_to_mel_matrix.shape[-1:]))
    log_mel = tf.math.log(mel_spectrograms + 1e-6)
    return log_mel

X_features = extract_mfe_features(X_raw).numpy()
print("Feature shape (samples, time_frames, mfe_bands):", X_features.shape)

### Step 4: Build & Train Lightweight 1D-CNN

In [ ]:
# Split train / validation
indices = np.arange(len(X_features))
np.random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

X_train, y_train = X_features[train_idx], y_raw[train_idx]
X_val, y_val = X_features[val_idx], y_raw[val_idx]

# Compact Conv1D Model for ESP32-S3 (< 25 KB Flash, < 12 KB RAM)
model = models.Sequential([
    layers.Input(shape=(X_features.shape[1], X_features.shape[2])),
    layers.Conv1D(16, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.25),
    layers.Dense(16, activation='relu'),
    layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    verbose=1
)

### Step 5: Quantize to INT8 (TFLite Micro with ESP-NN Hardware Acceleration)

In [ ]:
def representative_dataset():
    for sample in X_train[:100]:
        yield [np.expand_dims(sample, axis=0).astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open('drone_model_quant.tflite', 'wb') as f:
    f.write(tflite_quant_model)

print(f"Quantized Model Size: {len(tflite_quant_model) / 1024:.2f} KB")

### Step 6: Export to C Byte Array (`model_data.h`)
Downloads directly to your computer to drop into the ESP32-S3 firmware.

In [ ]:
# Convert .tflite to C array
hex_array = ', '.join([f'0x{b:02x}' for b in tflite_quant_model])
header_content = f"""#pragma once
#include <stdint.h>

const unsigned int g_drone_model_data_size = {len(tflite_quant_model)};
const unsigned char g_drone_model_data[] __attribute__((aligned(16))) = {{
    {hex_array}
}};
"""

with open('model_data.h', 'w') as f:
    f.write(header_content)

print("Exported 'model_data.h'. Drop this file directly into the ESP32-S3 project!")
try:
    from google.colab import files
    files.download('model_data.h')
except:
    pass